# LEAR MC62 -- 02 Staircase without Shims

**Segments:** Integral, Central
**Magnet order:** 1
**R_ref:** 0.033 m
### Part I: Setup & Data Quality
| # | Section |
|---|---------|
| 1 | Configuration & Imports |
| 2 | Kn Calibration |
| 3 | Run Discovery & Data Loading |
| 4 | Current Profile |
| 5 | cel/fed Safety Diagnostic |

### Part II: Pipeline Processing
| # | Section |
|---|---------|
| 6 | Pipeline Processing |
| 7 | Plateau Quality |

### Part III: Harmonic Analysis
| # | Section |
|---|---------|
| 8 | Main Field (B1) |
| 9 | b2 (Quadrupole) |
| 10 | b3 (Sextupole) |
| 11 | Higher Harmonics Overview |
| 12 | Multipole Spectrum |

### Part IV: Transfer Function & Inductance
| # | Section |
|---|---------|
| 13 | Transfer Function B1/I |
| 14 | Apparent vs Differential Inductance |

### Part V: Eddy Current & Settling
| # | Section |
|---|---------|
| 15 | Eddy Current Settling Analysis |
| 16 | Exponential Fits |
| 17 | Settling Bias Analysis |
| 18 | N_LAST Sensitivity Study |

### Part VI: Summary
| # | Section |
|---|---------|
| 19 | Comprehensive Statistics Table |
| 20 | Analysis Choices Summary |
| 21 | CSV Export |

---
## 1. Configuration & Imports

In [ ]:
# === CONFIGURATION ===
SEGMENT_CONFIGS = [
    {"name": "Integral", "kn_path": "MC62/2026-02-11/Kn values/Kn_R45_PCB_N1_0001_A_AC.txt", "merge_mode": "abs_upto_m_cmp_above", "is_fringe": False},
    {"name": "Central", "kn_path": "MC62/2026-02-11/Kn values/Kn_DQ_5_18_7_250_47x50_0001_A_AC.txt", "merge_mode": "abs_all", "is_fringe": False},
]
SEGMENTS = [s["name"] for s in SEGMENT_CONFIGS]

RUN_DIR_REL = "MC62/2026-02-11/02_staircase_without_shims/20260212_075344_staircase_without_shims_MC62/20260212_100000_MC62"
KN_PATHS = {"Integral": "MC62/2026-02-11/Kn values/Kn_R45_PCB_N1_0001_A_AC.txt", "Central": "MC62/2026-02-11/Kn values/Kn_DQ_5_18_7_250_47x50_0001_A_AC.txt"}

MAGNET_ORDER = 1
R_REF = 0.033
SAMPLES_PER_TURN = 1024

OPTIONS = ('dri', 'rot', 'cel', 'fed')
ENCODER_OFFSET_RAD = 3.141592653589793
MIN_B1_T = 1e-06
T_PER_TURN = 1.0  # 1 Hz rotation

N_LAST_TURNS = 170
N_SKIP_END = 0
N_SIGMA_CLIP = 5.0

print("LEAR MC62 -- 02 Staircase without Shims")
print("=" * 60)
print(f"  Segments      : {SEGMENTS}")
print(f"  Magnet order  : {MAGNET_ORDER}")
print(f"  R_ref         : {R_REF} m")
print(f"  Samples/turn  : {SAMPLES_PER_TURN}")
print(f"  Options       : {OPTIONS}")

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.optimize import curve_fit

%matplotlib widget
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 100,
})

REPO_ROOT = Path(".").resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "pyproject.toml").exists() or (REPO_ROOT / ".git").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rotating_coil_analyzer.analysis.kn_pipeline import load_segment_kn_txt
from rotating_coil_analyzer.analysis.utility_functions import (
    process_kn_pipeline,
    build_harmonic_rows,
    diagnose_cel_fed,
    mad_sigma_clip,
    eddy_model,
    fit_eddy_per_run,
    plateau_summary,
    discover_runs,
)
from rotating_coil_analyzer.ingest.channel_detect import robust_range

RUN_DIR = REPO_ROOT / "measurements" / RUN_DIR_REL

KN = {}
for _seg_name, _kn_rel in KN_PATHS.items():
    _kp = REPO_ROOT / "measurements" / _kn_rel
    assert _kp.exists(), f"Kn file not found: {_kp}"
    KN[_seg_name] = load_segment_kn_txt(str(_kp))

print(f"Repo root : {REPO_ROOT}")
print(f"Kn loaded : {list(KN.keys())}")
print("Imports ready.")

---
## 2. Kn Calibration

In [ ]:
for seg_name, kn_seg in KN.items():
    H = len(kn_seg.orders)
    print(f"\n{seg_name}: {H} harmonics")
    print(f"  Orders: {list(kn_seg.orders)}")
    kn_abs_n1 = abs(kn_seg.kn_abs[0])
    kn_cmp_n1 = abs(kn_seg.kn_cmp[0])
    ratio = kn_abs_n1 / max(kn_cmp_n1, 1e-30)
    print(f"  |Kn_abs(n=1)| = {kn_abs_n1:.6e}")
    print(f"  |Kn_cmp(n=1)| = {kn_cmp_n1:.6e}")
    print(f"  Abs/Cmp ratio (n=1): {ratio:.0f}x")

# Use first segment's Kn for harmonic count
_first_seg = SEGMENTS[0]
H = len(KN[_first_seg].orders)
Ns = SAMPLES_PER_TURN
m = MAGNET_ORDER
print(f"\nH={H}, Ns={Ns}, m={m}")

---
## 3. Run Discovery & Data Loading

Discover and load individual run files for all segments.

In [ ]:
runs = {}
for seg in SEGMENTS:
    runs[seg] = discover_runs(RUN_DIR, seg)
    # Classify ascending / descending branch
    for i, r in enumerate(runs[seg]):
        if i == 0 or r["I_nom"] >= runs[seg][i - 1]["I_nom"]:
            r["branch"] = "ascending"
        else:
            r["branch"] = "descending"
        if i > 0 and abs(r["I_nom"] - runs[seg][i - 1]["I_nom"]) < 1.0:
            r["branch"] = runs[seg][i - 1]["branch"]
    print(f"{seg}: {len(runs[seg])} runs discovered")
    for r in runs[seg][:3]:
        print(f"  Run {r['run_id']}: I_nom={r['I_nom']:.1f} A, {r['branch']}")
    if len(runs[seg]) > 3:
        print(f"  ... ({len(runs[seg]) - 3} more)")

---
## 4. Current Profile

Timeline showing all discovered runs.

In [ ]:
fig, axes = plt.subplots(1, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 5))
if len(SEGMENTS) == 1:
    axes = [axes]
for ax, seg in zip(axes, SEGMENTS):
    for r in runs[seg]:
        color = "tab:blue" if r["branch"] == "ascending" else "tab:red"
        ax.barh(r["I_nom"], 1, left=r["run_id"], height=20, color=color, alpha=0.7)
    ax.set_xlabel("Run index"); ax.set_ylabel("I_nom (A)")
    ax.set_title(f"Current Profile -- {seg}")
fig.suptitle("Run Discovery", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

---
## 5. cel/fed Safety Diagnostic

Run `diagnose_cel_fed()` on Integral high-current turns.

In [ ]:
# Run cel/fed diagnostic per segment
for seg in SEGMENTS:
    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    kn_seg = KN[seg]
    # Use first few runs at highest current
    _runs_sorted = sorted(runs[seg], key=lambda r: abs(r["I_nom"]), reverse=True)
    _hi_runs = _runs_sorted[:min(3, len(_runs_sorted))]
    if not _hi_runs:
        print(f"{seg}: no runs for cel/fed diagnostic")
        continue
    # Load turns from high-current runs
    _flux_abs, _flux_cmp, _t, _I = [], [], [], []
    for r in _hi_runs:
        raw = np.loadtxt(r["file"])
        _nt = raw.shape[0] // Ns
        _nk = _nt * Ns
        _flux_abs.append(raw[:_nk, 1].reshape(_nt, Ns))
        _flux_cmp.append(raw[:_nk, 2].reshape(_nt, Ns))
        _t.append(raw[:_nk, 0].reshape(_nt, Ns))
        _I.append(raw[:_nk, 3].reshape(_nt, Ns))
    _fa = np.concatenate(_flux_abs)[:100]
    _fc = np.concatenate(_flux_cmp)[:100]
    _tt = np.concatenate(_t)[:100]
    _ii = np.concatenate(_I)[:100]
    diag = diagnose_cel_fed(_fa, _fc, _tt, _ii, kn=kn_seg, r_ref=R_REF, magnet_order=MAGNET_ORDER)
    print(f"{seg} cel/fed: {diag.recommendation} -- {diag.reason}")
    if diag.recommendation == "UNSAFE":
        OPTIONS = tuple(o for o in OPTIONS if o not in ("cel", "fed"))
        print(f"  -> cel/fed disabled, OPTIONS = {OPTIONS}")

---
## 6. Pipeline Processing

Process each run through the Kn pipeline.

In [ ]:
df = {}
for seg in SEGMENTS:
    kn_seg = KN[seg]
    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    merge_mode = _scfg["merge_mode"]
    all_rows = []

    for r in runs[seg]:
        raw = np.loadtxt(r["file"])
        _nt = raw.shape[0] // Ns
        _nk = _nt * Ns

        _t = raw[:_nk, 0].reshape(_nt, Ns)
        _fa = raw[:_nk, 1].reshape(_nt, Ns)
        _fc = raw[:_nk, 2].reshape(_nt, Ns)
        _I = raw[:_nk, 3].reshape(_nt, Ns)

        result, C_merged, C_units, ok_main = process_kn_pipeline(
            flux_abs_turns=_fa, flux_cmp_turns=_fc,
            t_turns=_t, I_turns=_I,
            kn=kn_seg, r_ref=R_REF, magnet_order=m,
            options=OPTIONS, min_b1_T=MIN_B1_T,
            encoder_offset_rad=ENCODER_OFFSET_RAD,
            merge_mode=merge_mode,
        )
        extra = [
            {"run_id": r["run_id"], "I_nom": r["I_nom"], "branch": r["branch"],
              "turn_in_run": t, "segment": seg}
            for t in range(_nt)
        ]
        rows = build_harmonic_rows(result, C_merged, C_units, ok_main, m, extra)
        all_rows.extend(rows)

    df[seg] = pd.DataFrame(all_rows)
    print(f"{seg}: {len(df[seg])} turns from {len(runs[seg])} runs")

---
## 7. Plateau Quality

Compute per-run averages using the last N turns (settled).

In [ ]:
summ = {}
for seg in SEGMENTS:
    summ[seg] = plateau_summary(df[seg], N_LAST_TURNS, n_skip_end=N_SKIP_END)
    n_before = len(summ[seg])
    summ[seg], clip_info = mad_sigma_clip(summ[seg], "B1_mean", N_SIGMA_CLIP, label_col="branch")
    n_clipped = n_before - len(summ[seg])
    if n_clipped > 0:
        print(f"  {seg}: sigma clip removed {n_clipped} rows ({clip_info})")
    summ[seg]["TF_TperkA"] = summ[seg]["B1_mean"] / (summ[seg]["I_nom"] / 1000.0)
    print(f"{seg}: {len(summ[seg])} summary rows")
    print(summ[seg][["run_id", "I_nom", "branch", "B1_mean", "b2_units_mean", "b3_units_mean"]].head(10).to_string(index=False))

---
## 8. Main Field (B1)

In [ ]:
fig, axes = plt.subplots(1, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 5))
if len(SEGMENTS) == 1:
    axes = [axes]
for ax_idx, seg in enumerate(SEGMENTS):
    _df = df[seg]
    ok = _df["ok_main"]
    ax = axes[ax_idx]
    for branch, col in [("ascending", "tab:blue"), ("descending", "tab:red")]:
        mask = ok & (_df["branch"] == branch)
        ax.scatter(_df.loc[mask, "I_nom"], _df.loc[mask, "B1_T"],
                   s=8, alpha=0.5, color=col, label=branch)
    
    ax.set_xlabel("I (A)"); ax.set_ylabel("B1 (T)")
    ax.set_title(f"B1 -- {seg}"); ax.legend(fontsize=9)
fig.suptitle("Main Field (B1)", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

---
## 9. b2 (Quadrupole) -- first allowed harmonic error

In [ ]:
fig, axes = plt.subplots(1, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 5))
if len(SEGMENTS) == 1:
    axes = [axes]
for ax_idx, seg in enumerate(SEGMENTS):
    _df = df[seg]
    ok = _df["ok_main"]
    ax = axes[ax_idx]
    for branch, col in [("ascending", "tab:blue"), ("descending", "tab:red")]:
        mask = ok & (_df["branch"] == branch)
        ax.scatter(_df.loc[mask, "I_nom"], _df.loc[mask, "b2_units"],
                   s=8, alpha=0.5, color=col, label=branch)
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("I (A)"); ax.set_ylabel("b2 (units)")
    ax.set_title(f"b2 -- {seg}"); ax.legend(fontsize=9)
fig.suptitle("b2 (Quadrupole) -- first allowed harmonic error", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

---
## 10. b3 (Sextupole) -- first non-allowed harmonic

In [ ]:
fig, axes = plt.subplots(1, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 5))
if len(SEGMENTS) == 1:
    axes = [axes]
for ax_idx, seg in enumerate(SEGMENTS):
    _df = df[seg]
    ok = _df["ok_main"]
    ax = axes[ax_idx]
    for branch, col in [("ascending", "tab:blue"), ("descending", "tab:red")]:
        mask = ok & (_df["branch"] == branch)
        ax.scatter(_df.loc[mask, "I_nom"], _df.loc[mask, "b3_units"],
                   s=8, alpha=0.5, color=col, label=branch)
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("I (A)"); ax.set_ylabel("b3 (units)")
    ax.set_title(f"b3 -- {seg}"); ax.legend(fontsize=9)
fig.suptitle("b3 (Sextupole) -- first non-allowed harmonic", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

---
## 11. Higher Harmonics Overview

Statistics for all harmonics at key operating points (Integral).

In [ ]:
seg = "Integral"
if "file_discovery" == "file_discovery":
    _peak_mask = summ[seg]["I_nom"].abs() == summ[seg]["I_nom"].abs().max()
    _peak = summ[seg][_peak_mask]
    _bn_cols = sorted([c for c in summ[seg].columns if c.endswith("_mean")
                       and (c.startswith("b") or c.startswith("a"))
                       and c not in ("b2_units_mean", "b3_units_mean")],
                      key=lambda c: (c[0], int(c.split("_")[0][1:])))
    if _peak.empty or not _bn_cols:
        print("No peak-current data or harmonic columns found.")
    else:
        rows_tbl = []
        for col in _bn_cols:
            base = col.replace("_mean", "")
            std_col = base + "_std"
            mean_val = _peak[col].values[0]
            std_val = _peak[std_col].values[0] if std_col in _peak.columns else float("nan")
            rows_tbl.append({"Harmonic": base, "Mean [units]": mean_val, "Std [units]": std_val})
        _htable = pd.DataFrame(rows_tbl)
        print(f"Higher harmonics at peak |I| ({seg}):")
        print(_htable.to_string(index=False, float_format="%.3f"))
else:
    _df = results[seg]["df"]
    _summ = results[seg]["summ"]
    if not _summ.empty:
        _peak_mask = _summ["I_nom"].abs() == _summ["I_nom"].abs().max()
        _peak = _summ[_peak_mask]
        print(f"Higher harmonics at peak |I| ({seg}):")
        _bn_cols = sorted([c for c in _summ.columns if ("_mean" in c)
                           and (c.startswith("b") or c.startswith("a"))
                           and c not in ("b2_units_mean", "b3_units_mean", "B1_mean")],
                          key=lambda c: (c[0], int(c.split("_")[0][1:])))
        for col in _bn_cols[:20]:
            vals = _peak[col].values
            print(f"  {col}: {vals[0]:.4f}" if len(vals) > 0 else f"  {col}: --")

---
## 12. Multipole Spectrum

Bar charts of normal (bn) and skew (an) harmonics.

In [ ]:
seg = "Integral"
# Detect available harmonic orders from columns
if "file_discovery" == "file_discovery":
    _cols = df[seg].columns
else:
    _cols = results[seg]["df"].columns
bn_cols = [c for c in _cols if c.startswith("b") and c.endswith("_units") and c != "b1_units"]
orders = sorted([int(c.replace("b", "").replace("_units", "")) for c in bn_cols])

if orders:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    x = np.arange(len(orders))
    w = 0.35

    if "file_discovery" == "file_discovery":
        _src = df[seg][df[seg]["ok_main"]]
    else:
        _src = results[seg]["df"][results[seg]["df"]["ok_main"]]

    bn_means = [_src[f"b{nn}_units"].mean() for nn in orders]
    an_means = [_src[f"a{nn}_units"].mean() for nn in orders]

    ax = axes[0]
    ax.bar(x - w/2, bn_means, w, label="bn", color="steelblue", alpha=0.8)
    ax.bar(x + w/2, an_means, w, label="an", color="tab:orange", alpha=0.8)
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels(orders)
    ax.set_xlabel("n"); ax.set_ylabel("Units")
    ax.set_title("Spectrum (linear)"); ax.legend(fontsize=8)

    ax = axes[1]
    ax.bar(x - w/2, np.abs(bn_means), w, label="|bn|", color="steelblue", alpha=0.8)
    ax.bar(x + w/2, np.abs(an_means), w, label="|an|", color="tab:orange", alpha=0.8)
    ax.set_yscale("log")
    ax.set_xticks(x); ax.set_xticklabels(orders)
    ax.set_xlabel("n"); ax.set_ylabel("|Units|")
    ax.set_title("Spectrum (log)"); ax.legend(fontsize=8)

    fig.suptitle("Multipole Spectrum", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("No harmonic columns found for spectrum plot.")

---
## 13. Transfer Function B1/I

TF = B1 / I (units: T/kA).

In [ ]:
fig, axes = plt.subplots(1, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 5))
if len(SEGMENTS) == 1:
    axes = [axes]
for ax_idx, seg in enumerate(SEGMENTS):
    if "file_discovery" == "file_discovery":
        _df = df[seg][df[seg]["ok_main"]]
    else:
        _df = results[seg]["df"][results[seg]["df"]["ok_main"]]
    _df_tf = _df.copy()
    _df_tf["TF"] = _df_tf["B1_T"] / (_df_tf["I_mean_A"] / 1000.0)
    for branch, col in [("ascending", "tab:blue"), ("descending", "tab:red")]:
        mask = _df_tf["branch"] == branch
        axes[ax_idx].scatter(_df_tf.loc[mask, "I_nom"].abs(), _df_tf.loc[mask, "TF"],
                             s=8, alpha=0.5, color=col, label=branch)
    axes[ax_idx].set_xlabel("|I| (A)"); axes[ax_idx].set_ylabel("TF (T/kA)")
    axes[ax_idx].set_title(f"TF -- {seg}"); axes[ax_idx].legend(fontsize=9)
fig.suptitle("Transfer Function B1/I", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

---
## 14. Apparent vs Differential Inductance

**L_app** = B1/I, **L_d** = dB1/dI (from paired current levels).

In [ ]:
print("Inductance analysis for staircase data:")
for seg in SEGMENTS:
    if "file_discovery" == "file_discovery": _s = summ[seg]
    else: _s = results[seg]["summ"]
    if _s.empty:
        continue
    _s_ok = _s[_s["B1_mean"].notna()].copy()
    _s_ok["TF"] = _s_ok["B1_mean"] / (_s_ok["I_nom"] / 1000.0)
    for branch in ["ascending", "descending"]:
        sub = _s_ok[_s_ok["branch"] == branch].sort_values("I_nom")
        if len(sub) >= 2:
            # Compute Ld between consecutive steps
            dB = np.diff(sub["B1_mean"].values)
            dI = np.diff(sub["I_nom"].values) / 1000.0
            Ld = dB / dI
            valid = np.abs(dI) > 0.001
            if valid.any():
                print(f"  {seg} {branch}: Ld range = {Ld[valid].min():.4f} .. {Ld[valid].max():.4f} T/kA")

---
## 15. Eddy Current Settling Analysis

Turn-by-turn B1 for all runs. Eddy currents cause exponential decay.

In [ ]:
from rotating_coil_analyzer.analysis.utility_functions import fit_eddy_per_run

MIN_I_FOR_FIT = 10.0
T_PER_TURN = 1.0  # 1 Hz rotation

# Use first segment for eddy analysis
_eddy_seg = SEGMENTS[0]
df_eddy = df[_eddy_seg].copy()
print(f"Eddy analysis: {len(df_eddy)} turns, {df_eddy['run_id'].nunique()} runs")

### Raw B1 settling curves

In [ ]:
_I_noms = sorted(df_eddy["I_nom"].unique())
if len(_I_noms) == 0:
    print("No eddy data to plot")
else:
    cmap = plt.cm.coolwarm
    _norm_c = plt.Normalize(min(_I_noms), max(_I_noms))

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for sign_idx, (sign_label, sign_cond) in enumerate([("Positive current", lambda x: x > 0), ("Negative current", lambda x: x < 0)]):
        ax = axes[sign_idx]
        for run_id in sorted(df_eddy["run_id"].unique()):
            rdf = df_eddy[(df_eddy["run_id"] == run_id) & df_eddy["ok_main"]]
            if rdf.empty: continue
            I_nom = rdf["I_nom"].iloc[0]
            if not sign_cond(I_nom): continue
            ax.plot(rdf["turn_in_run"], rdf["B1_T"], lw=0.5,
                    color=cmap(_norm_c(I_nom)), alpha=0.7)
        ax.set_ylabel("B1 [T]"); ax.set_title(sign_label)
    axes[-1].set_xlabel("Turn in run")
    fig.suptitle("Raw settling curves -- B1 vs turn number", y=1.01)
    fig.tight_layout(); plt.show()

---
## 16. Exponential Fits

Fit single-exponential eddy model per run/supercycle.

In [ ]:
_fit_results = []
for run_id in sorted(df_eddy["run_id"].unique()):
    rdf = df_eddy[(df_eddy["run_id"] == run_id) & df_eddy["ok_main"]].sort_values("turn_in_run")
    if rdf.empty: continue
    I_nom = rdf["I_nom"].iloc[0]
    if abs(I_nom) < MIN_I_FOR_FIT: continue
    branch = rdf["branch"].iloc[0]
    I_mean = rdf["I_mean"].values if "I_mean" in rdf.columns else None
    res = fit_eddy_per_run(
        turns=rdf["turn_in_run"].values.astype(float),
        B1=rdf["B1_T"].values,
        run_id=run_id, I_nom=I_nom, branch=branch, I_mean=I_mean,
    )
    _fit_results.append(res)

_cols = ["run_id", "I_nom", "branch", "B_inf", "A", "tau", "tau_err",
         "tau_s", "tau_err_s", "r2", "n_turns", "quality", "reason"]
if _fit_results:
    df_fits_all = pd.DataFrame([
        {"run_id": r.run_id, "I_nom": r.I_nom, "branch": r.branch,
         "B_inf": r.B_inf, "A": r.A, "tau": r.tau, "tau_err": r.tau_err,
         "tau_s": r.tau * T_PER_TURN, "tau_err_s": r.tau_err * T_PER_TURN,
         "r2": r.r2, "n_turns": r.n_turns, "quality": r.quality, "reason": r.reason}
        for r in _fit_results
    ])
else:
    df_fits_all = pd.DataFrame(columns=_cols)

df_fits = df_fits_all[df_fits_all["quality"] == "GOOD"].copy()
print(f"Fits: {len(df_fits)} GOOD / {len(df_fits_all)} total")
if len(df_fits_all) > 0:
    print(f"{'Run':>4s} {'I [A]':>8s} {'tau [s]':>8s} {'R2':>8s} {'Quality':>12s}")
    print("-" * 50)
    for _, r in df_fits_all.iterrows():
        print(f"{r['run_id']:4.0f} {r['I_nom']:+8.1f} {r['tau_s']:8.2f} {r['r2']:8.4f} {r['quality']:>12s}")
else:
    print("No eddy fits produced (no qualifying runs)")

### Tau vs Current

In [ ]:
_branch_colors = {"ascending": "tab:blue", "descending": "tab:red"}
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
for branch, col in _branch_colors.items():
    mask = df_fits["branch"] == branch
    ax.scatter(df_fits.loc[mask, "I_nom"].abs(), df_fits.loc[mask, "tau_s"],
               c=col, label=branch, s=30, alpha=0.7)
ax.set_xlabel("|I| [A]"); ax.set_ylabel("tau [s]")
ax.set_title("Settling time constant vs current"); ax.legend()
plt.tight_layout(); plt.show()

if len(df_fits) > 0:
    print(f"Tau range: {df_fits['tau_s'].min():.1f} -- {df_fits['tau_s'].max():.1f} s")

---
## 17. Settling Bias Analysis

How b2/b3 averages change with averaging window.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
_show_runs = [r for _, r in df_fits.iterrows()]
_show_runs = _show_runs[::max(1, len(_show_runs) // 8)]

for quantity, col_name, ax, ylabel in [
    ("B1", "B1_T", axes[0], "B1 deviation [1e-4 rel.]"),
    ("b2", "b2_units", axes[1], "b2 deviation [units]"),
    ("b3", "b3_units", axes[2], "b3 deviation [units]"),
]:
    for row in _show_runs:
        run_id = row["run_id"]
        rdf = df_eddy[(df_eddy["run_id"] == run_id) & df_eddy["ok_main"]].sort_values("turn_in_run")
        if rdf.empty or col_name not in rdf.columns: continue
        vals = rdf[col_name].values
        ref = vals[-N_LAST_TURNS:].mean() if len(vals) >= N_LAST_TURNS else vals[-10:].mean()
        if quantity == "B1":
            dev = (vals - ref) / abs(ref) * 1e4 if abs(ref) > 1e-9 else vals - ref
        else:
            dev = vals - ref
        ax.plot(rdf["turn_in_run"].values, dev, lw=0.5, alpha=0.6)
    ax.set_ylabel(ylabel); ax.axhline(0, color="k", lw=0.5)
axes[-1].set_xlabel("Turn in run")
fig.suptitle("Settling bias", y=1.01); fig.tight_layout(); plt.show()

---
## 18. N_LAST Sensitivity Study

Scan N_LAST and show convergence.

In [ ]:
_n_last_values = np.arange(20, 331, 5)
_study_runs = df_fits["run_id"].unique()

_results_sweep = []
for _nl in _n_last_values:
    _B1_errs = []
    for run_id in _study_runs:
        rdf = df_eddy[(df_eddy["run_id"] == run_id) & df_eddy["ok_main"]].sort_values("turn_in_run")
        if len(rdf) < _nl + 20: continue
        B1_true = rdf["B1_T"].values[-20:].mean()
        B1_est = rdf["B1_T"].values[-_nl:].mean()
        _B1_errs.append((B1_est - B1_true) / abs(B1_true) * 1e4)
    if _B1_errs:
        _results_sweep.append({"N_LAST": _nl, "B1_bias": np.mean(_B1_errs), "B1_std": np.std(_B1_errs)})

_df_sweep = pd.DataFrame(_results_sweep)
if not _df_sweep.empty:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.fill_between(_df_sweep["N_LAST"],
                    _df_sweep["B1_bias"] - _df_sweep["B1_std"],
                    _df_sweep["B1_bias"] + _df_sweep["B1_std"], alpha=0.2, color="tab:blue")
    ax.plot(_df_sweep["N_LAST"], _df_sweep["B1_bias"], ".-", color="tab:blue")
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(N_LAST_TURNS, color="red", ls="--", lw=1, label=f"N_LAST = {N_LAST_TURNS}")
    ax.set_xlabel("N_LAST"); ax.set_ylabel("B1 bias [1e-4 rel.]")
    ax.set_title("B1 systematic bias vs averaging window"); ax.legend()
    plt.tight_layout(); plt.show()

---
## 19. Comprehensive Statistics Table

In [ ]:
print("=" * 70)
print("LEAR MC62 -- 02 Staircase without Shims")
print("=" * 70)

for seg in SEGMENTS:
    if "file_discovery" == "file_discovery":
        _s = summ[seg]
    else:
        _s = results[seg]["summ"]
    if _s.empty:
        print(f"\n{seg}: no summary data")
        continue
    print(f"\n--- {seg} ---")
    print(f"  Summary rows: {len(_s)}")
    if "I_nom" in _s.columns:
        print(f"  I range: {_s['I_nom'].min():.1f} .. {_s['I_nom'].max():.1f} A")
    if "B1_mean" in _s.columns:
        print(f"  B1 range: {_s['B1_mean'].min():.6f} .. {_s['B1_mean'].max():.6f} T")

---
## 20. Analysis Choices Summary

Document all analysis parameters for reproducibility.

In [ ]:
import datetime
print("ANALYSIS CHOICES")
print("=" * 60)
print(f"Generated    : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Title        : LEAR MC62 -- 02 Staircase without Shims")
print(f"Segments     : {SEGMENTS}")
print(f"Magnet order : {MAGNET_ORDER}")
print(f"R_ref        : {R_REF} m")
print(f"Samples/turn : {SAMPLES_PER_TURN}")
print(f"OPTIONS      : {OPTIONS}")
print(f"MIN_B1_T     : {MIN_B1_T}")
print(f"N_SIGMA_CLIP : {N_SIGMA_CLIP}")

---
## 21. CSV Export

In [ ]:
out_dir = REPO_ROOT / "output" / "MC62/02_without_shims"
out_dir.mkdir(parents=True, exist_ok=True)

for seg in SEGMENTS:
    fname = f"MC62_{seg}_all_turns.csv"
    df[seg].to_csv(out_dir / fname, index=False)
    print(f"Wrote {out_dir / fname}  ({len(df[seg])} rows)")

    fname_s = f"MC62_{seg}_summary.csv"
    summ[seg].to_csv(out_dir / fname_s, index=False)
    print(f"Wrote {out_dir / fname_s}  ({len(summ[seg])} rows)")

print("\nDone.")